# WAF HGNN (Colab Notebook)

UHG-enhanced HGNN pipeline for enterprise WAF (Splunk export) request embeddings and IP-level anomaly scoring.

**Before running:** set Colab runtime to **GPU** (Runtime → Change runtime type → GPU).

Upload WAF CSV splits to Drive (placeholder paths in `Config`). Production scoring uses **ALLOW** rows only; **BLOCK** rows are retained for evaluation.

## 1. Setup

In [ ]:
import torch

TORCH_VERSION = torch.__version__.split("+")[0]
CUDA_VERSION = "cu" + torch.version.cuda.replace(".", "") if torch.cuda.is_available() else "cpu"

# Install PyG wheels matched to Colab's preinstalled PyTorch/CUDA
!pip install -q torch-scatter torch-sparse torch-geometric -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+{CUDA_VERSION}.html
!pip install -q geoopt umap-learn scikit-learn tqdm matplotlib

In [ ]:
import json
import os
import time
from dataclasses import dataclass
from urllib.parse import urlparse

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse
import torch
import torch.nn as nn
import torch.nn.functional as F
import geoopt.optim
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import StandardScaler
from torch_scatter import scatter_mean
from tqdm.auto import tqdm
import umap

In [ ]:
# Splunk WAF export schema (normalized to snake_case)
WAF_SCHEMA_COLUMNS = [
    "event_time", "timestamp", "client_ip", "request_host", "webacl_id",
    "http_source_id", "http_source_name", "http_method", "http_version", "scheme",
    "uri", "query_string", "query_present", "action", "terminating_rule_id",
    "terminating_rule_type", "ja3", "ja4", "user_agent", "ua_family", "referer",
    "referer_present", "cookie_present", "accept_language_present",
    "accept_encoding_present", "connection_value", "header_count", "suspicious_path",
]

# Kept for aggregation / filtering; excluded from model feature matrix
PIPELINE_COLUMNS = ["event_time", "timestamp", "client_ip", "action", "suspicious_path"]

# Raw high-cardinality fields replaced by derived or bucketed features
EXCLUDE_FROM_FEATURES = [
    "client_ip", "user_agent", "referer", "query_string",
    "event_time", "timestamp", "action", "suspicious_path",
]


@dataclass
class Config:
    DATA_DIR: str = "/content/drive/MyDrive/waf/"
    ARTIFACT_DIR: str = "/content/drive/MyDrive/waf_hgnn_artifacts/"
    TRAIN_FILE: str = "train.csv"
    VAL_FILE: str = "val.csv"
    TEST_FILE: str = "test.csv"
    data_percentage: float = 1.0  # ~85k rows; set <1.0 to subsample for smoke tests
    score_action: str = "ALLOW"  # production scoring path
    k: int = 3
    hidden_channels: int = 64
    out_channels: int = 32
    num_layers: int = 3
    lr: float = 0.001
    weight_decay: float = 0.0005
    spread_weight: float = 0.1
    neg_multiplier: int = 2
    num_epochs: int = 100
    patience: int = 15
    log_every: int = 10
    scheduler_step: int = 10
    scheduler_gamma: float = 0.8
    pca_max_components: int = 50
    dbscan_eps: float = 0.5
    dbscan_min_samples: int = 5
    anomaly_percentile: float = 95.0
    USE_GPU: bool = True
    REBUILD_KNN: bool = True  # rebuild each run by default; set False to use cache
    RUN_UMAP_DBSCAN: bool = False  # optional exploration in section 8


cfg = Config()

In [ ]:
from google.colab import drive

mount_point = "/content/drive"

if not os.path.ismount(mount_point):
    if os.path.isdir(mount_point) and os.listdir(mount_point):
        print(
            f"Warning: '{mount_point}' exists and is not empty but is not mounted. "
            "Clearing stale contents before mount."
        )
        !rm -rf {mount_point}/*
    drive.mount(mount_point)
else:
    print(f"Google Drive already mounted at {mount_point}. Re-mounting with force_remount=True.")
    drive.mount(mount_point, force_remount=True)

os.makedirs(cfg.ARTIFACT_DIR, exist_ok=True)

if cfg.USE_GPU and torch.cuda.is_available():
    device = torch.device("cuda")
    torch.set_float32_matmul_precision("high")
else:
    device = torch.device("cpu")
    if cfg.USE_GPU:
        print("WARNING: GPU requested but not available. Enable GPU in Colab runtime.")

print(f"Using device: {device}")

## 2. WAF data load, feature engineering, ALLOW filter for scoring

Expected scale: ~85,412 rows (ALLOW ~57,432, BLOCK ~27,980). Train/val/test on **ALLOW** only; BLOCK rows loaded separately for section 8 eval.

In [ ]:
def normalize_waf_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Map Splunk export headers to snake_case WAF_SCHEMA_COLUMNS."""
    rename_map = {}
    for col in df.columns:
        key = col.strip().lower().replace(" ", "_")
        rename_map[col] = key
    out = df.rename(columns=rename_map)
    missing = [c for c in WAF_SCHEMA_COLUMNS if c not in out.columns]
    if missing:
        print(f"Warning: missing expected columns: {missing}")
    return out


def derive_uri_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "uri" not in out.columns:
        out["uri_path_length"] = 0
        out["uri_path_depth"] = 0
        return out

    def path_stats(uri):
        if pd.isna(uri) or not str(uri).strip():
            return 0, 0
        parsed = urlparse(str(uri))
        path = parsed.path or "/"
        depth = len([p for p in path.split("/") if p])
        return len(path), depth

    stats = out["uri"].apply(path_stats)
    out["uri_path_length"] = stats.apply(lambda x: x[0])
    out["uri_path_depth"] = stats.apply(lambda x: x[1])
    return out


def load_split(filename: str, frac: float, seed: int = 42) -> pd.DataFrame:
    path = os.path.join(cfg.DATA_DIR, filename)
    df = pd.read_csv(path, low_memory=False)
    df = normalize_waf_columns(df)
    df = derive_uri_features(df)
    if frac < 1.0:
        df = df.sample(frac=frac, random_state=seed).reset_index(drop=True)
    return df


def split_scoring_and_eval(df: pd.DataFrame):
    """ALLOW rows for HGNN scoring; retain BLOCK for evaluation."""
    if "action" not in df.columns:
        print("action column missing; using full dataframe for scoring")
        return df.copy(), pd.DataFrame(columns=df.columns)
    action = df["action"].astype(str).str.upper()
    scoring = df[action == cfg.score_action.upper()].reset_index(drop=True)
    eval_rows = df[action != cfg.score_action.upper()].reset_index(drop=True)
    return scoring, eval_rows


def build_feature_frame(df: pd.DataFrame) -> pd.DataFrame:
    drop_cols = [c for c in EXCLUDE_FROM_FEATURES if c in df.columns]
    feat = df.drop(columns=drop_cols, errors="ignore").copy()
    if "uri" in feat.columns:
        feat = feat.drop(columns=["uri"])
    non_numeric = feat.select_dtypes(exclude=[np.number]).columns.tolist()
    if non_numeric:
        feat = pd.get_dummies(feat, columns=non_numeric)
    return feat


def prepare_features(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame):
    train_x = build_feature_frame(train_df)
    val_x = build_feature_frame(val_df)
    test_x = build_feature_frame(test_df)

    val_x = val_x.reindex(columns=train_x.columns, fill_value=0)
    test_x = test_x.reindex(columns=train_x.columns, fill_value=0)

    train_means = train_x.mean(numeric_only=True)
    train_x = train_x.fillna(train_means)
    val_x = val_x.fillna(train_means)
    test_x = test_x.fillna(train_means)

    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_x)
    val_scaled = scaler.transform(val_x)
    test_scaled = scaler.transform(test_x)

    n_components = min(cfg.pca_max_components, train_scaled.shape[1])
    if n_components < train_scaled.shape[1]:
        pca = PCA(n_components=n_components)
        train_scaled = pca.fit_transform(train_scaled)
        val_scaled = pca.transform(val_scaled)
        test_scaled = pca.transform(test_scaled)
        print(f"PCA applied: {n_components} components")
    else:
        pca = None
        print("PCA skipped: feature count below cap")

    return train_scaled, val_scaled, test_scaled, list(train_x.columns), pca


def to_uhg_space(x: np.ndarray) -> np.ndarray:
    return np.concatenate([x, np.ones((x.shape[0], 1), dtype=x.dtype)], axis=1)


train_raw = load_split(cfg.TRAIN_FILE, cfg.data_percentage)
val_raw = load_split(cfg.VAL_FILE, cfg.data_percentage)
test_raw = load_split(cfg.TEST_FILE, cfg.data_percentage)

train_df, train_block = split_scoring_and_eval(train_raw)
val_df, val_block = split_scoring_and_eval(val_raw)
test_df, test_block = split_scoring_and_eval(test_raw)

block_eval_df = pd.concat([train_block, val_block, test_block], ignore_index=True)

print(f"Raw splits: train={train_raw.shape}, val={val_raw.shape}, test={test_raw.shape}")
print(
    f"Scoring ({cfg.score_action}): train={train_df.shape}, val={val_df.shape}, test={test_df.shape}"
)
print(f"BLOCK eval pool: {block_eval_df.shape}")
if "action" in train_raw.columns:
    print(train_raw["action"].astype(str).str.upper().value_counts().to_string())
if "suspicious_path" in train_raw.columns:
    print("\nsuspicious_path counts (all splits combined):")
    all_raw = pd.concat([train_raw, val_raw, test_raw], ignore_index=True)
    print(all_raw.groupby(all_raw["action"].astype(str).str.upper())["suspicious_path"].value_counts().to_string())

train_np, val_np, test_np, feature_columns, pca = prepare_features(train_df, val_df, test_df)
train_uhg = to_uhg_space(train_np)
val_uhg = to_uhg_space(val_np)
test_uhg = to_uhg_space(test_np)

train_features = torch.as_tensor(train_uhg, dtype=torch.float32, device=device)
val_features = torch.as_tensor(val_uhg, dtype=torch.float32, device=device)
test_features = torch.as_tensor(test_uhg, dtype=torch.float32, device=device)

print(f"Feature dims: {train_np.shape[1]} | UHG dims: {train_uhg.shape[1]}")

## 3. KNN graph construction

Rebuild each run by default (`REBUILD_KNN=True`). Optional cache with metadata validation for fast reruns.

In [ ]:
def build_knn_graph(data: np.ndarray, k: int) -> scipy.sparse.csr_matrix:
    return kneighbors_graph(
        data,
        n_neighbors=k,
        mode="connectivity",
        include_self=False,
        n_jobs=-1,
    )


def knn_to_edge_index(knn_graph: scipy.sparse.csr_matrix) -> torch.Tensor:
    coo = knn_graph.tocoo()
    return torch.tensor(np.vstack([coo.row, coo.col]), dtype=torch.long)


def knn_cache_metadata(name: str, data: np.ndarray) -> dict:
    return {
        "name": name,
        "dataset": "waf",
        "score_action": cfg.score_action,
        "num_nodes": int(data.shape[0]),
        "num_features": int(data.shape[1]),
        "k": cfg.k,
        "data_percentage": cfg.data_percentage,
    }


def load_or_build_edge_index(name: str, data: np.ndarray) -> torch.Tensor:
    cache_path = os.path.join(cfg.ARTIFACT_DIR, f"{name}_edge_index.pt")
    meta_path = os.path.join(cfg.ARTIFACT_DIR, f"{name}_edge_index.meta.json")
    expected_meta = knn_cache_metadata(name, data)

    if os.path.exists(cache_path) and not cfg.REBUILD_KNN:
        if os.path.exists(meta_path):
            with open(meta_path) as f:
                cached_meta = json.load(f)
            if cached_meta != expected_meta:
                print(
                    f"Cache metadata mismatch for {name}: {cached_meta} != {expected_meta}. "
                    "Set cfg.REBUILD_KNN = True to rebuild."
                )
            else:
                t0 = time.perf_counter()
                edge_index = torch.load(cache_path, map_location="cpu")
                elapsed = time.perf_counter() - t0
                print(
                    f"Loaded cached {name} edge index: {edge_index.shape[1]} edges "
                    f"({elapsed:.2f}s)"
                )
                return edge_index.to(device)
        else:
            edge_index = torch.load(cache_path, map_location="cpu")
            print(f"Loaded cached {name} edge index (no metadata file): {edge_index.shape[1]} edges")
            return edge_index.to(device)

    print(f"Building KNN graph for {name} ({data.shape[0]} nodes)...")
    t0 = time.perf_counter()
    knn_graph = build_knn_graph(data, cfg.k)
    edge_index = knn_to_edge_index(knn_graph)
    elapsed = time.perf_counter() - t0
    torch.save(edge_index.cpu(), cache_path)
    with open(meta_path, "w") as f:
        json.dump(expected_meta, f, indent=2)
    print(f"Saved {name} edge index: {edge_index.shape[1]} edges ({elapsed:.1f}s)")
    return edge_index.to(device)


train_edge_index = load_or_build_edge_index("train", train_np)
val_edge_index = load_or_build_edge_index("val", val_np)
test_edge_index = load_or_build_edge_index("test", test_np)

## 4. HGNN model and UHG loss

In [ ]:
def uhg_inner_product(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    return torch.sum(a[:, :-1] * b[:, :-1], dim=-1) - a[:, -1] * b[:, -1]


def uhg_norm(a: torch.Tensor) -> torch.Tensor:
    return torch.sum(a[:, :-1] ** 2, dim=-1) - a[:, -1] ** 2


def uhg_quadrance(a: torch.Tensor, b: torch.Tensor, eps: float = 1e-9) -> torch.Tensor:
    dot_product = uhg_inner_product(a, b)
    denom = torch.clamp((uhg_norm(a) * uhg_norm(b)).abs(), min=eps)
    return torch.clamp(1 - (dot_product ** 2) / denom, min=0.0)


def sample_negative_edges(
    num_nodes: int,
    num_neg: int,
    device: torch.device,
    generator: torch.Generator | None = None,
) -> torch.Tensor:
    if generator is None:
        return torch.randint(0, num_nodes, (2, num_neg), device=device)
    return torch.randint(0, num_nodes, (2, num_neg), device=device, generator=generator)


class GraphSAGE_CustomScatter(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, append_uhg: bool = True):
        super().__init__()
        self.append_uhg = append_uhg
        self.linear = nn.Linear(in_channels * 2, out_channels)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        source_nodes = edge_index[0]
        target_nodes = edge_index[1]
        source_features = x[source_nodes]
        agg = scatter_mean(source_features, target_nodes, dim=0, dim_size=x.size(0))
        out = self.linear(torch.cat([x, agg], dim=1))
        out = F.relu(out)
        if self.append_uhg:
            out = torch.cat([out, torch.ones((out.size(0), 1), device=out.device)], dim=1)
        return out


class HGNN_CustomScatter(nn.Module):
    def __init__(self, in_channels: int, hidden_channels: int, out_channels: int, num_layers: int):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(GraphSAGE_CustomScatter(in_channels, hidden_channels, append_uhg=True))
        for _ in range(num_layers - 2):
            self.convs.append(GraphSAGE_CustomScatter(hidden_channels + 1, hidden_channels, append_uhg=True))
        self.convs.append(GraphSAGE_CustomScatter(hidden_channels + 1, out_channels, append_uhg=False))

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        for conv in self.convs:
            x = conv(x, edge_index)
        return x


def uhg_loss(
    z: torch.Tensor,
    edge_index: torch.Tensor,
    spread_weight: float = 0.1,
    neg_multiplier: int = 2,
    generator: torch.Generator | None = None,
    return_parts: bool = False,
):
    pos_edge_index = edge_index
    if pos_edge_index.size(1) == 0:
        zero = torch.tensor(0.0, device=z.device)
        parts = {"link_loss": zero, "spread_loss": zero, "total_loss": zero}
        return parts if return_parts else zero

    quad = uhg_quadrance(z[pos_edge_index[0]], z[pos_edge_index[1]])
    pos_score = -quad

    num_neg = max(pos_edge_index.size(1) * neg_multiplier, 1)
    neg_edge_index = sample_negative_edges(z.size(0), num_neg, z.device, generator)
    neg_score = -uhg_quadrance(z[neg_edge_index[0]], z[neg_edge_index[1]])

    pos_loss = F.binary_cross_entropy_with_logits(pos_score, torch.ones_like(pos_score))
    neg_loss = F.binary_cross_entropy_with_logits(neg_score, torch.zeros_like(neg_score))
    link_loss = pos_loss + neg_loss
    spread_loss = quad.mean()
    total_loss = link_loss + spread_weight * spread_loss

    if return_parts:
        return {
            "link_loss": link_loss,
            "spread_loss": spread_loss,
            "total_loss": total_loss,
        }
    return total_loss


model = HGNN_CustomScatter(
    in_channels=train_features.shape[1],
    hidden_channels=cfg.hidden_channels,
    out_channels=cfg.out_channels,
    num_layers=cfg.num_layers,
).to(device)

optimizer = geoopt.optim.RiemannianAdam(
    model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay
)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, step_size=cfg.scheduler_step, gamma=cfg.scheduler_gamma
)

val_loss_generator = torch.Generator(device=device)
val_loss_generator.manual_seed(42)

## 5. Training with early stopping

Early-stop on **val_link** (BCE link loss), not total loss with spread regularizer.

In [ ]:
@torch.no_grad()
def evaluate(model: nn.Module, features: torch.Tensor, edge_index: torch.Tensor) -> dict:
    model.eval()
    out = model(features, edge_index)
    parts = uhg_loss(
        out,
        edge_index,
        cfg.spread_weight,
        cfg.neg_multiplier,
        generator=val_loss_generator,
        return_parts=True,
    )
    return {k: float(v.item()) for k, v in parts.items()}


def train_with_early_stopping(
    model: nn.Module,
    train_features: torch.Tensor,
    train_edge_index: torch.Tensor,
    val_features: torch.Tensor,
    val_edge_index: torch.Tensor,
    optimizer,
    scheduler,
) -> dict:
    best_val = float("inf")
    best_epoch = 0
    stale_epochs = 0
    checkpoint_path = os.path.join(cfg.ARTIFACT_DIR, "best_model.pt")
    history = {"train_loss": [], "val_loss": [], "val_link_loss": [], "val_spread_loss": []}

    for epoch in tqdm(range(1, cfg.num_epochs + 1), desc="Training"):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        out = model(train_features, train_edge_index)
        loss = uhg_loss(out, train_edge_index, cfg.spread_weight, cfg.neg_multiplier)
        loss.backward()
        optimizer.step()
        scheduler.step()

        train_loss = float(loss.item())
        val_parts = evaluate(model, val_features, val_edge_index)
        val_link = val_parts["link_loss"]
        val_total = val_parts["total_loss"]
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_total)
        history["val_link_loss"].append(val_link)
        history["val_spread_loss"].append(val_parts["spread_loss"])

        if val_link < best_val:
            best_val = val_link
            best_epoch = epoch
            stale_epochs = 0
            torch.save(
                {
                    "model": model.state_dict(),
                    "epoch": epoch,
                    "val_link_loss": val_link,
                    "val_total_loss": val_total,
                },
                checkpoint_path,
            )
        else:
            stale_epochs += 1

        if epoch == 1 or epoch % cfg.log_every == 0:
            lr = scheduler.get_last_lr()[0]
            print(
                f"Epoch {epoch:3d} | train={train_loss:.4f} | "
                f"val_link={val_link:.4f} | val_total={val_total:.4f} | "
                f"val_spread={val_parts['spread_loss']:.4f} | "
                f"best_link={best_val:.4f} @ {best_epoch} | lr={lr:.6f}"
            )

        if stale_epochs >= cfg.patience:
            print(f"Early stopping at epoch {epoch} (best val_link={best_val:.4f} @ epoch {best_epoch})")
            break

    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model"])
    print(
        f"Loaded best checkpoint from epoch {checkpoint['epoch']} "
        f"(val_link={checkpoint['val_link_loss']:.4f}, val_total={checkpoint['val_total_loss']:.4f})"
    )
    return history


history = train_with_early_stopping(
    model,
    train_features,
    train_edge_index,
    val_features,
    val_edge_index,
    optimizer,
    scheduler,
)

## 6. Embeddings and IP-level anomaly aggregation

In [ ]:
def embedding_distance_scores(embeddings: np.ndarray) -> np.ndarray:
    """Per-request anomaly score = distance to embedding centroid."""
    clean = np.nan_to_num(embeddings, nan=0.0)
    centroid = clean.mean(axis=0, keepdims=True)
    return np.linalg.norm(clean - centroid, axis=1)


model.eval()
with torch.no_grad():
    train_embeddings = model(train_features, train_edge_index).cpu().numpy()
    test_embeddings = model(test_features, test_edge_index).cpu().numpy()

train_scores = embedding_distance_scores(train_embeddings)
test_scores = embedding_distance_scores(test_embeddings)
score_threshold = float(np.percentile(train_scores, cfg.anomaly_percentile))

request_anomalies = pd.DataFrame({
    "client_ip": train_df["client_ip"].values if "client_ip" in train_df.columns else "unknown",
    "anomaly_score": train_scores,
    "is_anomaly": train_scores >= score_threshold,
})
if "suspicious_path" in train_df.columns:
    request_anomalies["suspicious_path"] = train_df["suspicious_path"].values

ip_anomalies = (
    request_anomalies.groupby("client_ip", dropna=False)
    .agg(
        request_count=("anomaly_score", "size"),
        mean_score=("anomaly_score", "mean"),
        max_score=("anomaly_score", "max"),
        anomaly_requests=("is_anomaly", "sum"),
    )
    .reset_index()
    .sort_values("max_score", ascending=False)
)
ip_anomalies["anomaly_rate"] = ip_anomalies["anomaly_requests"] / ip_anomalies["request_count"]

embeddings_path = os.path.join(cfg.ARTIFACT_DIR, "embeddings.npy")
np.save(embeddings_path, train_embeddings)
request_anomalies.to_csv(os.path.join(cfg.ARTIFACT_DIR, "request_anomalies.csv"), index=False)
ip_anomalies.to_csv(os.path.join(cfg.ARTIFACT_DIR, "ip_anomalies.csv"), index=False)

print(f"Saved embeddings: {train_embeddings.shape} -> {embeddings_path}")
print(f"Anomaly threshold (p{cfg.anomaly_percentile}): {score_threshold:.4f}")
print(f"Flagged requests: {request_anomalies['is_anomaly'].sum()} / {len(request_anomalies)}")
print("\nTop IPs by max anomaly score:")
print(ip_anomalies.head(10).to_string(index=False))

## 7. Blocklist output stub (WAF API placeholder)

In [ ]:
BLOCKLIST_PATH = os.path.join(cfg.ARTIFACT_DIR, "blocklist_candidates.json")

# Stub: IPs with high anomaly rate and multiple flagged requests
blocklist_candidates = ip_anomalies[
    (ip_anomalies["anomaly_requests"] >= 3) & (ip_anomalies["anomaly_rate"] >= 0.5)
][["client_ip", "request_count", "anomaly_requests", "anomaly_rate", "max_score"]]

payload = {
    "generated_at": pd.Timestamp.utcnow().isoformat(),
    "source": "waf_hgnn_colab",
    "score_action": cfg.score_action,
    "anomaly_percentile": cfg.anomaly_percentile,
    "candidates": blocklist_candidates.to_dict(orient="records"),
    "waf_api_stub": {
        "endpoint": "https://YOUR_WAF_API/v1/ip-blocklist",
        "method": "POST",
        "note": "Replace with enterprise WAF blocklist API call",
    },
}

with open(BLOCKLIST_PATH, "w") as f:
    json.dump(payload, f, indent=2)

print(f"Wrote {len(blocklist_candidates)} blocklist candidates -> {BLOCKLIST_PATH}")
print(json.dumps(payload["waf_api_stub"], indent=2))

## 8. Optional evaluation (BLOCK rows, suspicious_path, UMAP+DBSCAN)

In [ ]:
print("=== BLOCK row pool (held out from scoring) ===")
if block_eval_df.empty:
    print("No BLOCK rows loaded.")
else:
    print(f"BLOCK rows: {block_eval_df.shape}")
    if "suspicious_path" in block_eval_df.columns:
        print(block_eval_df["suspicious_path"].value_counts().to_string())
    if "terminating_rule_type" in block_eval_df.columns:
        print("\nTop terminating_rule_type:")
        print(block_eval_df["terminating_rule_type"].value_counts().head(10).to_string())

print("\n=== ALLOW scoring set: suspicious_path cross-check ===")
if "suspicious_path" in request_anomalies.columns:
    flagged = request_anomalies[request_anomalies["is_anomaly"]]
    baseline = request_anomalies["suspicious_path"].mean()
    flagged_rate = flagged["suspicious_path"].mean() if len(flagged) else float("nan")
    print(f"Baseline suspicious_path rate: {baseline:.1%}")
    print(f"Flagged-request suspicious_path rate: {flagged_rate:.1%}")
else:
    print("suspicious_path not available in scoring set.")

if cfg.RUN_UMAP_DBSCAN:
    print("\n=== Optional UMAP + DBSCAN (exploration) ===")
    clean = np.nan_to_num(train_embeddings, nan=0.0)
    reducer = umap.UMAP(
        n_neighbors=15,
        min_dist=0.1,
        n_components=2,
        metric="euclidean",
        n_jobs=-1,
        low_memory=True,
        random_state=42,
    )
    umap_2d = reducer.fit_transform(clean)
    labels = DBSCAN(eps=cfg.dbscan_eps, min_samples=cfg.dbscan_min_samples).fit_predict(umap_2d)
    plt.figure(figsize=(10, 7))
    plt.scatter(umap_2d[:, 0], umap_2d[:, 1], c=labels, cmap="tab20", s=8, alpha=0.6)
    plt.title("UMAP + DBSCAN (ALLOW scoring set)")
    plt.colorbar()
    plt.grid(True, alpha=0.3)
    plt.show()
    if "suspicious_path" in train_df.columns:
        sp = train_df["suspicious_path"].to_numpy()
        for cid in np.unique(labels):
            mask = labels == cid
            if mask.sum() == 0:
                continue
            rate = sp[mask].mean()
            if rate > sp.mean() * 1.5:
                print(f"Cluster {cid}: size={mask.sum()}, suspicious_path rate={rate:.1%}")
else:
    print("\nUMAP+DBSCAN skipped (set cfg.RUN_UMAP_DBSCAN=True to enable).")